# BirdCLEF 2026 - Test Mel File Naming
Quick test to verify mel filename generation before processing all files

In [ ]:
import os, json, math, random, gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
print("✅ Imports successful")

In [ ]:
# === DATA PATHS ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"

df = pd.read_csv(TRAIN_META_CSV)
species = sorted(df["primary_label"].astype(str).unique())
species_set = set(species)

print(f"Number of species: {len(species)}")
print(f"Total train files: {len(df)}")

idx = {lab: i for i, lab in enumerate(species)}
print("✅ Data loaded")

In [ ]:
CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
)

print(f"Config: sr={CFG['sr']}, n_mels={CFG['n_mels']}, fmin={CFG['fmin']}")

In [ ]:
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S01 = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
    return S01.astype(np.float32)

print("✅ Helper functions defined")

In [ ]:
# TEST: Sample from train_audio
MEL_OUT_DIR = "/kaggle/working/mels_test"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

print("Sample: Processing 5 files from train_audio...")
print("=" * 70)

train_audio_mels = []
for idx_, row in df.head(5).iterrows():
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])

    mel_filename = row["filename"].replace("/", "_") + ".npy"
    mel_path = Path(MEL_OUT_DIR) / mel_filename
    np.save(mel_path, mel)
    train_audio_mels.append(mel_filename)
    print(f"Saved: {mel_filename}")

print(f"\nGenerated {len(train_audio_mels)} train_audio mel files")

In [ ]:
# TEST: Sample from train_soundscapes
print("\n" + "=" * 70)
print("Sample: Processing 5 files from train_soundscapes...")
print("=" * 70)

try:
    soundscape_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    
    # Get species that appear in soundscapes
    soundscape_species = set()
    for species_list in soundscape_labels['primary_label']:
        species_list = str(species_list).split(';')
        for sp in species_list:
            soundscape_species.add(sp.strip())
    
    # Find species in soundscapes but NOT in train_audio
    missing_species = soundscape_species - species_set
    print(f"Species missing from train_audio: {len(missing_species)}")
    
    if len(missing_species) > 0:
        # Helper function to convert HH:MM:SS to seconds
        def time_string_to_seconds(time_str):
            """Convert HH:MM:SS format to seconds"""
            parts = str(time_str).split(':')
            if len(parts) == 3:
                hours, minutes, seconds = map(int, parts)
                return hours * 3600 + minutes * 60 + seconds
            return 0
        
        SOUNDSCAPES_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
        soundscape_mels = []
        sample_count = 0
        
        # Only process first 5 from soundscapes
        for idx_, row in soundscape_labels.iterrows():
            if sample_count >= 5:
                break
                
            filename = row['filename']
            start_time_str = row['start']
            end_time_str = row['end']
            species_list = str(row['primary_label']).split(';')
            species_list = [sp.strip() for sp in species_list]
            
            # Only process if contains a missing species
            if not any(sp in missing_species for sp in species_list):
                continue
            
            audio_path = Path(SOUNDSCAPES_DIR) / filename
            if not audio_path.exists():
                continue
            
            try:
                y, sr0 = sf.read(audio_path, always_2d=False)
            except:
                continue
            
            # Convert time strings (HH:MM:SS) to seconds
            start_time_sec = time_string_to_seconds(start_time_str)
            end_time_sec = time_string_to_seconds(end_time_str)
            
            # Extract segment
            start_sample = int(start_time_sec * sr0)
            end_sample = int(end_time_sec * sr0)
            segment = y[start_sample:end_sample]
            
            # Resample if needed
            if sr0 != CFG["sr"]:
                segment = librosa.resample(segment, orig_sr=sr0, target_sr=CFG["sr"])
            
            # Enforce 5 seconds
            segment = fixed_length_mono(segment, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(segment, CFG["sr"])
            
            # Create filename with STRIPPED extension
            mel_name = f"soundscape_{filename.rsplit(\".\", 1)[0]}_{start_time_str.replace(':', '')}_{end_time_str.replace(':', '')}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)
            soundscape_mels.append(mel_name)
            print(f"Saved: {mel_name}")
            sample_count += 1
        
        print(f"\nGenerated {len(soundscape_mels)} soundscape mel files")
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# SUMMARY & VERIFICATION
print("\n" + "=" * 70)
print("FILENAME VERIFICATION")
print("=" * 70)

print(f"\nTrain audio mels ({len(train_audio_mels)}):")
for fname in train_audio_mels:
    print(f"  {fname}")

if 'soundscape_mels' in locals():
    print(f"\nSoundscape mels ({len(soundscape_mels)}):")
    for fname in soundscape_mels:
        print(f"  {fname}")

# Check if files exist
print("\n" + "=" * 70)
print("FILE EXISTENCE CHECK")
print("=" * 70)
all_mels = train_audio_mels + (soundscape_mels if 'soundscape_mels' in locals() else [])
for mel_name in all_mels:
    mel_path = Path(MEL_OUT_DIR) / mel_name
    exists = mel_path.exists()
    status = "EXISTS" if exists else "MISSING"
    print(f"  [{status}] {mel_name}")

print(f"\nTotal mel files created: {len(all_mels)}")
print("\nNameing verification complete!")